>> Neka je dat pozitivan realan broj _x_ sa konačnim decimalskim zapisom i neka su dati prirodni brojevi _n_ i _m_, tako da _n<m_. Formirati niz razlomaka _p/q_ takvih da za imenilac _q_ važi _n<=q<=m_ (tj. _q=n,n+1,...,m_) i pri tom imeniocu _q_ pridružujemo brojilac _p_ koji određujemo zaokruživanjem na najbliži priodan broj proizvoda _x⋅q_. Predstaviti svaki razlomak _p/q_ u obliku verižnog razlomaka. U nizu razlomaka _p/q_ izdvojiti:
>> * najbolje racionalne aproksimacije I vrste,
>> * najbolje racionalne aproksimacije II vrste,
>> * sortirati sve razlomke _p/q_ po uslovu minimalnosti apsolutne greske _|x-p/q|_.

Napisani kod ćemo testirati na primjeru konstante _v=log₂(3)-1=0.5849625007211561815..._, a za parametre _n_ i _m_ uzeti brojeve 7 i 53 respektivno.

In [6]:
from math import log

v = log(3, 2) - 1
n, m = 7, 53

Prvi korak je pisanje pomoćnih funkcija, koje će olakšati generisanje verižnog oblika razlomaka i utvrđivanje "kvaliteta" razlomaka.

Prva funkcija (_generate_continued_fraction_) će generisati verižni oblik razlomka za dato _p_ i _q_.
Naredne dvije funkcije (_is_fst_deg_best_ i _is_snd_deg_best_) utvrđuju da li se radi o najboljoj racionalnoj aproksimaciji I i II vrste.

In [7]:
def generate_simple_continued_fraction(p: int, q: int) -> list[int]:
	a = [0]
	a[0], p = divmod(p, q)

	while p != 0:
		quotient, remainder = divmod(q, p)
		a.append(quotient)
		p, q = remainder, p

	return a


def is_fst_deg_best(x: float, p: int, q: int) -> bool:
	for s in range(1, q):
		r = round(s * x)
		if r / s == p / q:
			continue

		if abs(x - r / s) < abs(x - p / q):
			return False

	return True

def is_snd_deg_best(x: float, p: int, q: int) -> bool:
	for s in range(1, q):
		r = round(s * x)
		if r / s == p / q:
			continue

		if abs(s * x - r) < abs(q * x - p):
			return False

	return True

Nakon pisanja pomoćnih funkcija, možemo preći na dio koda koji prolazi kroz svako moguće _q_ i generiše razlomke. A usput, za svaki razlomak, utvrđujemo da li se radi o najboljoj racionalnoj aproksimaciji.

Sve razlomke ćemo čuvati u listi, kako bismo ih mogli kasnije i sortirati i ispisati.

In [8]:
from dataclasses import dataclass

@dataclass(order=True)
class RationalApproximation:
	p: int
	q: int
	a: list[int]

	first_degree: bool
	second_degree: bool

	def __float__(self) -> float:
		return self.p / self.q

fractions: list[RationalApproximation] = []
for q in range(n, m + 1):
	p = round(q * v)

	a = generate_simple_continued_fraction(p, q)
	first_degree = is_fst_deg_best(v, p, q)
	second_degree = is_snd_deg_best(v, p, q) if first_degree else False

	fractions.append(RationalApproximation(
		p, q,
		a, first_degree, second_degree
	))

Potom je potrebno isfiltrirati i sortirati razlomke. Prema tekstu zadatka, sortiranje se vrši po rastućoj vrijednosti apsolutne greške. Dok, kada je riječ o filtriranju, potrebno je ukloniti razlomke koji su u suštini isti, tj. one koji se pojavljuju i u nekom skraćenom obliku (npr. _7/12_ i _14/24_).

In [9]:
def simplify_fraction(p: int, q: int) -> tuple[int, int]:
    i = 2
    while i < min(p, q) + 1:
        if p % i == 0 and q % i == 0:
            p = p // i
            q = q // i
        else:
            i += 1
    return p, q

unique_fractions: dict[tuple[int, int], list[RationalApproximation]] = {}
for frac in fractions:
    simplified = simplify_fraction(frac.p, frac.q)
    if simplified not in unique_fractions:
        unique_fractions[simplified] = []
    unique_fractions[simplified].append(frac)

fractions = [min(group) for group in unique_fractions.values()]
fractions.sort(key=lambda r: abs(r.p / r.q - v))

Ssada imamo sve informacije da ispišemo traženu tabelu. Svaki razlomak je napisan u 3 oblika:
1. kao razlomak
2. kao realan broj
3. kroz verižni zapis

Različite boje određuju "kvalitet" aproksimacije:
- **Crvena** boja - Najbolja racionalna aproksimacija I vrste
- **Plava** boja - Najbolja racionalna aproksimacija II vrste
- **Bijela** boja - Ostali razlomci

In [10]:
from termcolor import colored

def format_simple_continued_fraction(a: list[int]) -> str:
	if len(a) == 1:
		return f"[{a[0]}]"

	return f"[{a[0]}; {', '.join(map(str, a[1:]))}]"

for fraction in fractions:
	if fraction.second_degree:
		color = "red"
	elif fraction.first_degree:
		color = "blue"
	else:
		color = "light_grey"

	ratio_format = f"{fraction.p}/{fraction.q}"
	real_format = f"{float(fraction):.9f}"
	difference = f"{v - float(fraction):+.9f}"
	array_format = format_simple_continued_fraction(fraction.a)

	text = f"{ratio_format} = {real_format} ({difference}), {array_format}"

	print(colored(text, color))

31/53 = 0.584905660 (+0.000056840), [0; 1, 1, 2, 2, 4]
24/41 = 0.585365854 (-0.000403353), [0; 1, 1, 2, 2, 3]
17/29 = 0.586206897 (-0.001244396), [0; 1, 1, 2, 2, 2]
7/12 = 0.583333333 (+0.001629167), [0; 1, 1, 2, 2]
27/46 = 0.586956522 (-0.001994021), [0; 1, 1, 2, 2, 1, 2]
10/17 = 0.588235294 (-0.003272793), [0; 1, 1, 2, 3]
25/43 = 0.581395349 (+0.003567152), [0; 1, 1, 2, 1, 1, 3]
18/31 = 0.580645161 (+0.004317339), [0; 1, 1, 2, 1, 1, 2]
23/39 = 0.589743590 (-0.004781089), [0; 1, 1, 2, 3, 2]
29/50 = 0.580000000 (+0.004962501), [0; 1, 1, 2, 1, 1, 1, 2]
13/22 = 0.590909091 (-0.005946590), [0; 1, 1, 2, 4]
11/19 = 0.578947368 (+0.006015132), [0; 1, 1, 2, 1, 2]
29/49 = 0.591836735 (-0.006874234), [0; 1, 1, 2, 4, 2]
26/45 = 0.577777778 (+0.007184723), [0; 1, 1, 2, 1, 2, 2]
16/27 = 0.592592593 (-0.007630092), [0; 1, 1, 2, 5]
15/26 = 0.576923077 (+0.008039424), [0; 1, 1, 2, 1, 3]
19/32 = 0.593750000 (-0.008787499), [0; 1, 1, 2, 6]
19/33 = 0.575757576 (+0.009204925), [0; 1, 1, 2, 1, 4]
22/37 = 